# Kerdos — MLflow Experiment Tracking

Task: Create an MLflow pipeline based on our final project, then zip the artifact of the experiment and submit in Binusmaya.

This notebook re-runs evaluation for **all 3 models** (Technical, Fundamental, Sentiment) using the **same trained models, features, and preprocessing** as the final deployed Kerdos project, and logs parameters, metrics, tags, and model artifacts to MLflow for each run.

- **Technical** — XGBoost Classifier + RF Regressor, loaded from `model_classifier.pkl` / `model_regressor.pkl`
- **Fundamental** — KMeans Clustering + RF Classifier, loaded from `fundamental_kmeans.pkl` / `fundamental_rf.pkl`
- **Sentiment** — TF-IDF + VADER + RF Classifier, loaded from `sentiment_rf.pkl` / `sentiment_tfidf.pkl`

Team: Hans Ewaldo Kristiawan / Christian Verrell / Adrian Marcello Budiman

In [ ]:
# ============================================================
# BLOCK 0 — INSTALL & IMPORTS
# ============================================================
!pip install -q mlflow scikit-learn xgboost imbalanced-learn joblib vaderSentiment

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import os
import json
import joblib
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    matthews_corrcoef, roc_auc_score,
    mean_absolute_error, r2_score,
)

REPO_ROOT = os.path.abspath(os.getcwd())

mlflow.set_experiment("Kerdos - AI Stock Analysis (Technical, Fundamental, Sentiment)")

print("✅ Setup done. Working dir:", REPO_ROOT)

---
## Run 1 — Technical Model (XGBoost Classifier + RF Regressor)

Loads `model_classifier.pkl`, `model_regressor.pkl`, `features.json`, and the cleaned AAPL dataset, re-evaluates on the held-out test split (last 20% chronologically — no shuffle), and logs everything to MLflow.

In [ ]:
# ============================================================
# BLOCK 1 — TECHNICAL MODEL: LOAD ARTIFACTS
# ============================================================

# --- Paths: update if your files live elsewhere ---
TECH_CSV_PATH        = os.path.join(REPO_ROOT, "aapl_raw.csv")
TECH_CLF_PKL_PATH    = os.path.join(REPO_ROOT, "model_classifier.pkl")
TECH_REG_PKL_PATH    = os.path.join(REPO_ROOT, "model_regressor.pkl")
TECH_FEATURES_PATH   = os.path.join(REPO_ROOT, "features.json")

clf_tech = joblib.load(TECH_CLF_PKL_PATH)
reg_tech = joblib.load(TECH_REG_PKL_PATH)

with open(TECH_FEATURES_PATH) as f:
    FEATURES = json.load(f)

print("✅ Technical model artifacts loaded")
print(f"   Classifier : {type(clf_tech).__name__}")
print(f"   Regressor  : {type(reg_tech).__name__}")
print(f"   Features   : {len(FEATURES)} columns")

In [ ]:
# ============================================================
# BLOCK 2 — TECHNICAL MODEL: REBUILD FEATURE SET FROM RAW CSV
# ============================================================
# Same engineer_features() logic as notebooks/Technical_ML.ipynb,
# the source of truth for model_classifier.pkl / model_regressor.pkl

def engineer_features(df):
    d = df.copy()
    d['HL_spread']      = d['High'] - d['Low']
    d['OC_spread']      = d['Close'] - d['Open']
    d['HL_pct']         = d['HL_spread'] / d['Close']
    d['Price_position'] = (d['Close'] - d['Low']) / d['HL_spread'].replace(0, np.nan)

    d['DayOfWeek']  = d['Date'].dt.dayofweek
    d['IsMonday']   = (d['DayOfWeek'] == 0).astype(int)
    d['IsFriday']   = (d['DayOfWeek'] == 4).astype(int)
    d['WeekOfYear'] = d['Date'].dt.isocalendar().week.astype(int)
    d['Month']      = d['Date'].dt.month

    for lag in [1, 2, 3, 5]:
        d[f'Close_lag{lag}']  = d['Close'].shift(lag)
        d[f'Return_lag{lag}'] = d['Close'].pct_change(lag)

    for w in [5, 10, 20, 50]:
        d[f'MA_{w}']       = d['Close'].rolling(w).mean()
        d[f'MA_ratio_{w}'] = d['Close'] / d[f'MA_{w}']

    d['Volatility_5d']  = d['Close'].pct_change(1).rolling(5).std()
    d['Volatility_10d'] = d['Close'].pct_change(1).rolling(10).std()
    d['Volatility_20d'] = d['Close'].pct_change(1).rolling(20).std()

    d['Volume_MA5']   = d['Volume'].rolling(5).mean()
    d['Volume_ratio'] = d['Volume'] / d['Volume_MA5']

    delta = d['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss.replace(0, np.nan)
    d['RSI_14'] = 100 - (100 / (1 + rs))

    ema12 = d['Close'].ewm(span=12, adjust=False).mean()
    ema26 = d['Close'].ewm(span=26, adjust=False).mean()
    d['MACD']        = ema12 - ema26
    d['MACD_signal'] = d['MACD'].ewm(span=9, adjust=False).mean()
    d['MACD_hist']   = d['MACD'] - d['MACD_signal']

    bb_mid = d['Close'].rolling(20).mean()
    bb_std = d['Close'].rolling(20).std()
    d['BB_upper'] = bb_mid + 2 * bb_std
    d['BB_lower'] = bb_mid - 2 * bb_std
    d['BB_width'] = (d['BB_upper'] - d['BB_lower']) / bb_mid
    d['BB_pct']   = (d['Close'] - d['BB_lower']) / (d['BB_upper'] - d['BB_lower'])

    d['Next_return'] = d['Close'].pct_change(1).shift(-1)

    def label_signal(r):
        if r > 0.007:   return 2
        elif r < -0.007: return 0
        else: return 1

    d['Signal'] = d['Next_return'].apply(lambda r: label_signal(r) if pd.notna(r) else np.nan)
    return d


df_tech = pd.read_csv(TECH_CSV_PATH)
df_tech['Date'] = pd.to_datetime(df_tech['Date'])
df_tech = df_tech.sort_values('Date').reset_index(drop=True)

df_tech = engineer_features(df_tech)
df_tech_clean = df_tech.dropna().reset_index(drop=True)

X_tech = df_tech_clean[FEATURES]
y_clf_tech = df_tech_clean['Signal'].astype(int)
y_reg_tech = df_tech_clean['Next_return']

# Same chronological 80/20 split as training (no shuffle — avoids leakage)
SPLIT = int(len(X_tech) * 0.8)
X_test_tech       = X_tech.iloc[SPLIT:]
y_clf_test_tech    = y_clf_tech.iloc[SPLIT:]
y_reg_test_tech    = y_reg_tech.iloc[SPLIT:]

print(f"✅ Technical test set rebuilt: {len(X_test_tech)} rows")

In [ ]:
# ============================================================
# BLOCK 3 — TECHNICAL MODEL: EVALUATE + LOG TO MLFLOW
# ============================================================

with mlflow.start_run(run_name="Technical_XGBoost_Classifier_RF_Regressor"):

    print("Technical Model — XGBoost Classifier + RF Regressor")

    mlflow.set_tags({
        "model_type":     "XGBoost Classifier + RF Regressor",
        "task":           "Technical Signal Prediction",
        "dataset":        "AAPL OHLCV via yfinance, 10 years (2,540 rows, 19 features)",
        "target":         "Signal {0=SELL, 1=HOLD, 2=BUY}",
        "split_strategy": "Chronological 80/20 (no shuffle) — preserves temporal order, avoids leakage",
        "balancing":      "SMOTE on training set only",
        "features":       f"{len(FEATURES)} engineered features (RSI, MACD, Bollinger Bands, lag returns, volatility, calendar)",
        "primary_metric": "ROC-AUC (macro OvR)",
        "source_notebook":"notebooks/Technical_ML.ipynb (source of truth for model_classifier.pkl / model_regressor.pkl)",
        "team":           "Kerdos - Hans/Verrell/Marcel",
    })

    mlflow.log_params({
        "n_features":     len(FEATURES),
        "classifier":     type(clf_tech).__name__,
        "regressor":      type(reg_tech).__name__,
        "test_size":      0.2,
    })
    # Log model hyperparameters if available (XGBoost / RandomForest expose get_params)
    try:
        mlflow.log_params({f"clf__{k}": v for k, v in clf_tech.get_params().items()
                            if isinstance(v, (int, float, str, bool)) or v is None})
    except Exception as e:
        print(f"   (skipped clf param logging: {e})")

    # --- Classifier evaluation ---
    y_pred_tech  = clf_tech.predict(X_test_tech)
    y_proba_tech = clf_tech.predict_proba(X_test_tech)

    acc      = accuracy_score(y_clf_test_tech, y_pred_tech)
    macro_f1 = f1_score(y_clf_test_tech, y_pred_tech, average='macro', zero_division=0)
    bal_acc  = balanced_accuracy_score(y_clf_test_tech, y_pred_tech)
    mcc      = matthews_corrcoef(y_clf_test_tech, y_pred_tech)
    try:
        roc_auc = roc_auc_score(
            pd.get_dummies(y_clf_test_tech).values, y_proba_tech,
            multi_class='ovr', average='macro'
        )
    except Exception:
        roc_auc = float('nan')

    # --- Regressor evaluation ---
    y_reg_pred_tech = reg_tech.predict(X_test_tech)
    mae = mean_absolute_error(y_reg_test_tech, y_reg_pred_tech)
    r2  = r2_score(y_reg_test_tech, y_reg_pred_tech)

    # --- Latency ---
    row = X_test_tech.iloc[[-1]]
    t0 = time.time()
    clf_tech.predict(row); clf_tech.predict_proba(row); reg_tech.predict(row)
    latency_ms = (time.time() - t0) * 1000

    test_acc_tech = acc

    mlflow.log_metrics({
        "test_accuracy":      acc,
        "test_macro_f1":      macro_f1,
        "test_balanced_acc":  bal_acc,
        "test_mcc":           mcc,
        "test_roc_auc_macro": roc_auc,
        "regressor_mae_pct":  mae * 100,
        "regressor_r2":       r2,
        "inference_latency_ms": latency_ms,
    })

    mlflow.xgboost.log_model(clf_tech, name="technical_classifier") if "XGB" in type(clf_tech).__name__ \
        else mlflow.sklearn.log_model(clf_tech, name="technical_classifier")
    mlflow.sklearn.log_model(reg_tech, name="technical_regressor")

    print(f"   Accuracy        : {acc:.4f}")
    print(f"   Macro F1        : {macro_f1:.4f}")
    print(f"   MCC             : {mcc:.4f}")
    print(f"   ROC-AUC (macro) : {roc_auc:.4f}")
    print(f"   Regressor MAE   : {mae*100:.4f}%")
    print(f"   Regressor R2    : {r2:.4f}")
    print(f"   Latency         : {latency_ms:.2f} ms")
    print("✅ Technical run logged to MLflow")

---
## Run 2 — Fundamental Model (KMeans Clustering + RF Classifier)

Loads `fundamental_kmeans.pkl`, `fundamental_scaler.pkl`, `fundamental_rf.pkl`, `fundamental_imputer.pkl`, and label/feature metadata, re-evaluates on a fresh stratified test split, and logs to MLflow.

In [ ]:
# ============================================================
# BLOCK 4 — FUNDAMENTAL MODEL: LOAD ARTIFACTS
# ============================================================

FUND_CSV_PATH          = os.path.join(REPO_ROOT, "fundamentals_raw.csv")
FUND_KMEANS_PKL_PATH   = os.path.join(REPO_ROOT, "fundamental_kmeans.pkl")
FUND_SCALER_PKL_PATH   = os.path.join(REPO_ROOT, "fundamental_scaler.pkl")
FUND_RF_PKL_PATH       = os.path.join(REPO_ROOT, "fundamental_rf.pkl")
FUND_IMPUTER_PKL_PATH  = os.path.join(REPO_ROOT, "fundamental_imputer.pkl")
FUND_LABELMAP_PATH     = os.path.join(REPO_ROOT, "fundamental_label_map.json")
FUND_ENCODER_PATH      = os.path.join(REPO_ROOT, "fundamental_label_encoder.json")
FUND_FEATURES_PATH     = os.path.join(REPO_ROOT, "fundamental_features.json")

kmeans_fund  = joblib.load(FUND_KMEANS_PKL_PATH)
scaler_fund  = joblib.load(FUND_SCALER_PKL_PATH)
rf_fund      = joblib.load(FUND_RF_PKL_PATH)
imputer_fund = joblib.load(FUND_IMPUTER_PKL_PATH)

with open(FUND_LABELMAP_PATH) as f:
    label_map_fund = json.load(f)
with open(FUND_ENCODER_PATH) as f:
    label_encoder_fund = json.load(f)
with open(FUND_FEATURES_PATH) as f:
    CLUSTER_FEATURES = json.load(f)

print("✅ Fundamental model artifacts loaded")
print(f"   KMeans clusters : {kmeans_fund.n_clusters}")
print(f"   RF classifier   : {type(rf_fund).__name__}")
print(f"   Features        : {CLUSTER_FEATURES}")

In [ ]:
# ============================================================
# BLOCK 5 — FUNDAMENTAL MODEL: REBUILD TEST SET
# ============================================================
# Reuses the same cleaning steps as notebooks/Fundamental_ML.ipynb:
# numeric coercion -> drop invalid PE -> impute -> log-transform skewed cols

df_fund = pd.read_csv(FUND_CSV_PATH)

for col in CLUSTER_FEATURES:
    df_fund[col] = pd.to_numeric(df_fund[col], errors='coerce')

df_model_fund = df_fund[['Ticker', 'Name', 'Sector'] + CLUSTER_FEATURES].copy()
df_model_fund = df_model_fund[
    (df_model_fund['PE_Ratio'].isna()) | (df_model_fund['PE_Ratio'] > 0)
].reset_index(drop=True)

df_model_fund[CLUSTER_FEATURES] = imputer_fund.transform(df_model_fund[CLUSTER_FEATURES])

# Log-transform skewed columns (same as training)
df_model_fund['MarketCap'] = np.log1p(df_model_fund['MarketCap'])
df_model_fund['PE_Ratio']  = np.log1p(df_model_fund['PE_Ratio'].clip(lower=0))
df_model_fund['PB_Ratio']  = np.log1p(df_model_fund['PB_Ratio'].clip(lower=0.01))

# Assign Valuation label via KMeans cluster -> label_map (same pipeline as training)
X_scaled_fund = scaler_fund.transform(df_model_fund[CLUSTER_FEATURES])
clusters_fund = kmeans_fund.predict(X_scaled_fund)
df_model_fund['Cluster']   = clusters_fund
df_model_fund['Valuation'] = pd.Series(clusters_fund).astype(str).map(label_map_fund).values
df_model_fund = df_model_fund.dropna(subset=['Valuation']).reset_index(drop=True)
df_model_fund['Label'] = df_model_fund['Valuation'].map(label_encoder_fund)

X_fund = scaler_fund.transform(df_model_fund[CLUSTER_FEATURES])
y_fund = df_model_fund['Label'].values

X_train_fund, X_test_fund, y_train_fund, y_test_fund = train_test_split(
    X_fund, y_fund, test_size=0.2, random_state=42, stratify=y_fund
)

print(f"✅ Fundamental test set rebuilt: {len(X_test_fund)} rows")
print(f"   Label distribution (full): \n{df_model_fund['Valuation'].value_counts()}")

In [ ]:
# ============================================================
# BLOCK 6 — FUNDAMENTAL MODEL: EVALUATE + LOG TO MLFLOW
# ============================================================

with mlflow.start_run(run_name="Fundamental_KMeans_RF_Classifier"):

    print("Fundamental Model — KMeans Clustering + RF Classifier")

    mlflow.set_tags({
        "model_type":     "KMeans Clustering + RF Classifier",
        "task":           "Valuation Classification (cluster then classify)",
        "dataset":        "800 global stocks via yfinance (1,000 rows incl. synthetic, 6 features)",
        "target":         "Valuation {Undervalued, Fair, Overvalued}",
        "split_strategy": "Stratified train/test split (80/20), preserves class ratio",
        "balancing":      "SMOTE with k_neighbors adjusted to minority class size",
        "clustering":     f"KMeans, K={kmeans_fund.n_clusters} (selected via elbow + silhouette score)",
        "primary_metric": "Balanced Accuracy",
        "source_notebook":"notebooks/Fundamental_ML.ipynb (source of truth for fundamental_*.pkl)",
        "team":           "Kerdos - Hans/Verrell/Marcel",
    })

    mlflow.log_params({
        "n_features":  len(CLUSTER_FEATURES),
        "n_clusters":  kmeans_fund.n_clusters,
        "classifier":  type(rf_fund).__name__,
        "test_size":   0.2,
    })
    try:
        mlflow.log_params({f"clf__{k}": v for k, v in rf_fund.get_params().items()
                            if isinstance(v, (int, float, str, bool)) or v is None})
    except Exception as e:
        print(f"   (skipped clf param logging: {e})")

    y_pred_fund = rf_fund.predict(X_test_fund)

    train_bal_acc = balanced_accuracy_score(y_train_fund, rf_fund.predict(X_train_fund))
    test_bal_acc  = balanced_accuracy_score(y_test_fund, y_pred_fund)
    test_acc      = accuracy_score(y_test_fund, y_pred_fund)
    test_mcc      = matthews_corrcoef(y_test_fund, y_pred_fund)
    gap           = train_bal_acc - test_bal_acc

    # Latency
    row = X_test_fund[-1:]
    t0 = time.time()
    rf_fund.predict(row); rf_fund.predict_proba(row)
    latency_ms = (time.time() - t0) * 1000

    mlflow.log_metrics({
        "train_balanced_accuracy": train_bal_acc,
        "test_balanced_accuracy":  test_bal_acc,
        "test_accuracy":           test_acc,
        "test_mcc":                test_mcc,
        "train_test_gap":          gap,
        "inference_latency_ms":    latency_ms,
    })

    mlflow.sklearn.log_model(rf_fund, name="fundamental_classifier")
    mlflow.sklearn.log_model(kmeans_fund, name="fundamental_kmeans")

    print(f"   Train Balanced Acc : {train_bal_acc:.4f}")
    print(f"   Test Balanced Acc  : {test_bal_acc:.4f}")
    print(f"   Train-Test Gap     : {gap:.4f}")
    print(f"   MCC                : {test_mcc:.4f}")
    print(f"   Latency            : {latency_ms:.2f} ms")
    print("✅ Fundamental run logged to MLflow")

---
## Run 3 — Sentiment Model (TF-IDF + VADER + RF Classifier)

Loads `sentiment_rf.pkl`, `sentiment_tfidf.pkl`, `sentiment_vader.pkl`, and the labeled headline dataset, re-evaluates on a fresh stratified test split, and logs to MLflow.

In [ ]:
# ============================================================
# BLOCK 7 — SENTIMENT MODEL: LOAD ARTIFACTS
# ============================================================

import re
from scipy.sparse import hstack, csr_matrix
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

SENT_CSV_PATH        = os.path.join(REPO_ROOT, "sentiment_training_data.csv")
SENT_RF_PKL_PATH     = os.path.join(REPO_ROOT, "sentiment_rf.pkl")
SENT_TFIDF_PKL_PATH  = os.path.join(REPO_ROOT, "sentiment_tfidf.pkl")
SENT_VADER_PKL_PATH  = os.path.join(REPO_ROOT, "sentiment_vader.pkl")
SENT_LABELMAP_PATH   = os.path.join(REPO_ROOT, "sentiment_label_map.json")

rf_sent    = joblib.load(SENT_RF_PKL_PATH)
tfidf_sent = joblib.load(SENT_TFIDF_PKL_PATH)
vader_sent = joblib.load(SENT_VADER_PKL_PATH)

with open(SENT_LABELMAP_PATH) as f:
    label_map_sent = json.load(f)

print("✅ Sentiment model artifacts loaded")
print(f"   Classifier : {type(rf_sent).__name__}")
print(f"   TF-IDF vocab size: {len(tfidf_sent.vocabulary_)}")
print(f"   Label map  : {label_map_sent}")

In [ ]:
# ============================================================
# BLOCK 8 — SENTIMENT MODEL: REBUILD TEST SET
# ============================================================
# Same preprocess_text() + VADER + TF-IDF feature combination as
# notebooks/Sentimental_ML.ipynb, the source of truth for sentiment_*.pkl

STOP_WORDS = set(stopwords.words('english'))
KEEP_WORDS = {'not','no','never','nor','neither','hardly','barely',
              'crash','surge','plunge','soar','fear','greed','bull','bear'}
STOP_WORDS = STOP_WORDS - KEEP_WORDS

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

def get_vader_features(text):
    scores = vader_sent.polarity_scores(text)
    return {
        'vader_pos': scores['pos'], 'vader_neg': scores['neg'],
        'vader_neu': scores['neu'], 'vader_compound': scores['compound'],
        'vader_pos_neg_ratio': scores['pos'] / (scores['neg'] + 1e-6),
        'vader_extreme': abs(scores['compound']),
    }

df_sent = pd.read_csv(SENT_CSV_PATH)
df_sent['text_clean'] = df_sent['text'].apply(preprocess_text)

vader_feats_sent = pd.DataFrame(list(df_sent['text'].apply(get_vader_features)))
X_tfidf_sent = tfidf_sent.transform(df_sent['text_clean'])
X_combined_sent = hstack([X_tfidf_sent, csr_matrix(vader_feats_sent.values)])
y_sent = df_sent['label'].values

X_train_sent, X_test_sent, y_train_sent, y_test_sent = train_test_split(
    X_combined_sent, y_sent, test_size=0.2, random_state=42, stratify=y_sent
)

print(f"✅ Sentiment test set rebuilt: {X_test_sent.shape[0]} rows, {X_test_sent.shape[1]} features")

In [ ]:
# ============================================================
# BLOCK 9 — SENTIMENT MODEL: EVALUATE + LOG TO MLFLOW
# ============================================================

with mlflow.start_run(run_name="Sentiment_TFIDF_VADER_RF_Classifier"):

    print("Sentiment Model — TF-IDF + VADER + RF Classifier")

    mlflow.set_tags({
        "model_type":     "TF-IDF + VADER + RF Classifier",
        "task":           "Market Sentiment Classification (NLP)",
        "dataset":        "Manually labeled financial headlines (120 rows, 206 features: 200 TF-IDF + 6 VADER)",
        "target":         "Sentiment {Fear, Neutral, Greed}",
        "split_strategy": "Stratified train/test split (80/20)",
        "balancing":      "class_weight='balanced' (handles Neutral=30 vs Fear/Greed=45 imbalance)",
        "feature_eng":    "TF-IDF (max_features=200, ngram 1-2) fit only on training set + VADER lexicon scores",
        "primary_metric": "Macro F1",
        "source_notebook":"notebooks/Sentimental_ML.ipynb (source of truth for sentiment_*.pkl)",
        "team":           "Kerdos - Hans/Verrell/Marcel",
    })

    mlflow.log_params({
        "tfidf_max_features": 200,
        "vader_features":     6,
        "classifier":         type(rf_sent).__name__,
        "test_size":          0.2,
    })
    try:
        mlflow.log_params({f"clf__{k}": v for k, v in rf_sent.get_params().items()
                            if isinstance(v, (int, float, str, bool)) or v is None})
    except Exception as e:
        print(f"   (skipped clf param logging: {e})")

    y_pred_sent = rf_sent.predict(X_test_sent)

    acc      = accuracy_score(y_test_sent, y_pred_sent)
    macro_f1 = f1_score(y_test_sent, y_pred_sent, average='macro', zero_division=0)
    mcc      = matthews_corrcoef(y_test_sent, y_pred_sent)
    bal_acc  = balanced_accuracy_score(y_test_sent, y_pred_sent)

    # Latency — single text inference end-to-end
    sample_text = "Markets crash as recession fears grip investors and panic selling emerges"
    t0 = time.time()
    sample_clean = preprocess_text(sample_text)
    sample_vader = pd.DataFrame([get_vader_features(sample_text)])
    sample_tfidf = tfidf_sent.transform([sample_clean])
    sample_X = hstack([sample_tfidf, csr_matrix(sample_vader.values)])
    rf_sent.predict(sample_X); rf_sent.predict_proba(sample_X)
    latency_ms = (time.time() - t0) * 1000

    mlflow.log_metrics({
        "test_accuracy":      acc,
        "test_macro_f1":      macro_f1,
        "test_mcc":           mcc,
        "test_balanced_acc":  bal_acc,
        "inference_latency_ms": latency_ms,
    })

    mlflow.sklearn.log_model(rf_sent, name="sentiment_classifier")

    print(f"   Accuracy   : {acc:.4f}")
    print(f"   Macro F1   : {macro_f1:.4f}")
    print(f"   MCC        : {mcc:.4f}")
    print(f"   Latency    : {latency_ms:.2f} ms")
    print("✅ Sentiment run logged to MLflow")

---
## Summary — All 3 Runs

In [ ]:
# ============================================================
# BLOCK 10 — PIPELINE SUMMARY
# ============================================================

print("  Kerdos — MLflow Pipeline Summary")
print("  Experiment : Kerdos - AI Stock Analysis")
print()
print(f"  Run 1 (Technical)   : XGBoost Classifier + RF Regressor")
print(f"                        Test Accuracy : {test_acc_tech:.4f}")
print()
print(f"  Run 2 (Fundamental) : KMeans + RF Classifier")
print(f"                        Test Balanced Acc : {test_bal_acc:.4f}")
print()
print(f"  Run 3 (Sentiment)   : TF-IDF + VADER + RF Classifier")
print(f"                        Test Macro F1 : {macro_f1:.4f}")
print()
print("✅ All 3 runs logged. Zip the 'mlruns/' folder (and mlflow.db if present)")
print("   and submit as artifacts.zip to Binusmaya.")